In [8]:
import torch
import re
import random
from torch import nn
import math
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter

##### utils实现

In [9]:
#vocab的实现，用于将字母转为数字，将数字转为字母
class Vocab:
    def __init__(self, tokens):
        """
        参数:
            tokens: 所有字符的列表，例如 ['a', 'b', 'c', ...]
        """
        # 去重并排序
        unique_tokens = sorted(set(tokens))
        
        # 建立索引映射
        self.idx_to_token = unique_tokens
        self.token_to_idx = {token: idx for idx, 
                             token in enumerate(unique_tokens)}
    
    def __getitem__(self, token):
        """支持 vocab[token] 语法，返回 token 对应的索引"""
        return self.token_to_idx[token]


    def __len__(self):
        """返回词表大小"""
        return len(self.idx_to_token)
#定义梯度裁剪
def grad_clipping(net, theta):  #@save
    """裁剪梯度"""
    ##获取参数param
    ##如果是继承了nn.Moudule
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params

    ##如果参数的和超过了θ，所有的参数都×
    ##θ/norm
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

##### 封装LSTM

In [10]:
##获取参数
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    def three():
        return [normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device)]
    W_xi,W_hi,b_i = three()
    W_xf,W_hf,b_f = three()
    W_xo,W_ho,b_o = three()
    W_xc,W_hc,b_c = three()
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xi,W_hi,b_i
              ,W_xf,W_hf,b_f,
               W_xo,W_ho,b_o 
               ,W_xc,W_hc,b_c,
               W_hq,b_q]

    for param in params:
        param.requires_grad_(True)
    return params
##初始化隐藏向量H和C
def init_lstm_state(batch_size, num_hiddens, device):
    return (
        torch.zeros((batch_size, num_hiddens), device=device),  # H
        torch.zeros((batch_size, num_hiddens), device=device),  # C
    )
#LSTM运算过程
def lstm(inputs, state, params):
    (W_xi,W_hi,b_i,
     W_xf,W_hf,b_f,
     W_xo,W_ho,b_o,
     W_xc,W_hc,b_c,
     W_hq,b_q) = params
    H,C = state
    outputs = []

    for X in inputs:
        F = torch.sigmoid(torch.mm(X,W_xf)+torch.mm(H,W_hf)+b_f)
        I = torch.sigmoid(torch.mm(X,W_xi)+torch.mm(H,W_hi)+b_i)
        C_candidate = torch.tanh(torch.mm(X,W_xc)+torch.mm(H,W_hc)+b_c)
        O = torch.sigmoid(torch.mm(X,W_xo)+torch.mm(H,W_ho)+b_o)
        C = F*C + I*C_candidate
        H = torch.tanh(C)*O
        Y = torch.mm(H,W_hq)+b_q
        outputs.append(Y)

    return torch.cat(outputs, dim=0), (H,C)
class LSTM: 
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn

    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)

##### 封装文本数据加载器

In [11]:
#读取并清洗数据集
def read_time_machine(file_path='timemachine.txt'):
    """读取时间机器数据集，并做基本文本清洗"""

    ##打开文件，取每一行
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # 只保留英文字母，将连续空白合并为一个空格，并转换为小写
    lines = [
        re.sub(r'[^A-Za-z]+', ' ', line).strip().lower()
        for line in lines
    ]

    # 去掉清洗后的空行
    return [line for line in lines if line]
#加载数据集，返回词表对象和数字索引列表
def load_corpus_time_machine(file_path='timemachine.txt'):
    """
    返回：
        corpus: 文本中每个字符对应的词表索引
        vocab:  字符词表
    """
    #max_token用于限制数据量，加快训练速度
    lines = read_time_machine(file_path)

    # 将所有行连接起来，并保留单词之间的空格
    text = ' '.join(lines)

    # 字符级建模
    tokens = list(text)

    vocab = Vocab(tokens)
    corpus = [vocab[token] for token in tokens]
    return corpus, vocab
#采样器，获得输入X和输出Y，形状[batch_size, num_steps]
def seq_data_iter_random(corpus, batch_size, num_steps):
    """
    使用随机采样生成小批量序列。

    X 和 Y 的形状均为：
        [batch_size, num_steps]

    Y 是 X 向后移动一个字符得到的标签。
    """
    # 随机选择起点
    ##使得每次切分的起点不同，增加样本多样性
    ##从0到numstep-1
    offset = random.randint(0, num_steps - 1)
    corpus = corpus[offset:]

    ##每个随机取的corpus可以构成多少个满足时间步的序列
    num_subseqs = (len(corpus) - 1) // num_steps

    # 每个子序列的起始位置
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    ##对起始位置序列进行打乱
    random.shuffle(initial_indices)

    def data(pos):
        return corpus[pos:pos + num_steps]

    ##获得的时间步序列能有几个batchsize
    num_batches = num_subseqs // batch_size

    for i in range(0, batch_size * num_batches, batch_size):
        ##在起始序列中拿batchsize个
        batch_indices = initial_indices[i:i + batch_size]

        ##获得X和Y的序列
        X = [data(j) for j in batch_indices]
        Y = [data(j + 1) for j in batch_indices]
    

        ##yield不断返回batchsizeX和Y的tensor的向量
        ##一次返回一个batchsize，直到下回调用
        yield (
            torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long)
        )
##封装数据加载器
class SeqDataLoader:
    """时间机器数据集的字符级序列批量加载器"""

    def __init__(self,
                 batch_size,
                 num_steps,
                 file_path='timemachine.txt'):
        self.corpus, self.vocab = load_corpus_time_machine(
            file_path=file_path
        )

        self.batch_size = batch_size
        self.num_steps = num_steps

    def __iter__(self):
        return seq_data_iter_random(
            self.corpus,
            self.batch_size,
            self.num_steps
        )
##创建迭代器，返回迭代器和词表对照对象
def load_data_time_machine(batch_size,
                           num_steps,
                           file_path='timemachine.txt'):
    """创建数据迭代器和词表"""
    data_iter = SeqDataLoader(
        batch_size=batch_size,
        num_steps=num_steps,
        file_path=file_path,
    )

    return data_iter, data_iter.vocab

##### 训练函数封装

In [12]:
#定义预测函数
def predict_ch8(prefix, 
                num_preds, net, vocab, device):
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))

    for y in prefix[1:]:  # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])

    for _ in range(num_preds):  # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])
#单次训练循环
def train_epoch_lstm(net,
                    data_iter,
                    loss_fn,
                    optimizer,
                    device,
                    clip_theta=1.0):

    total_loss = 0.0
    total_tokens = 0
    for X, Y in data_iter:
        X = X.to(device)
        Y = Y.to(device)

        batch_size = X.shape[0]

        #每次初始化隐变量状态
        state = net.begin_state(
            batch_size=batch_size,
            device=device
        )
        optimizer.zero_grad()

        # 前向传播
        y_hat, state = net(X, state)
        y = Y.T.reshape(-1)
        loss = loss_fn(y_hat, y)

        # 反向传播
        loss.backward()

        grad_clipping(net, clip_theta)

        # SGD 参数更新
        optimizer.step()

        num_tokens = y.numel()

        #一轮的总损失
        total_loss += loss.item() * num_tokens
        #总共预测的数量
        total_tokens += num_tokens

    average_loss = total_loss / total_tokens
    #限制困惑度数值，防止数值溢出
    perplexity = math.exp(min(average_loss, 20))

    return average_loss, perplexity
def train_lstm(net,
              data_iter,
              vocab,
              device,
              num_epochs,
              log_dir,
              learning_rate=0.01,
              clip_theta=1.0,
              predict_prefix='time traveller',
              num_preds=20):
    loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        net.params,
        lr=learning_rate
    )
    writer = SummaryWriter(log_dir=log_dir)
    for epoch in range(1, num_epochs + 1):
        average_loss, perplexity = train_epoch_lstm(
            net=net,
            data_iter=data_iter,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device,
            clip_theta=clip_theta
        )
        print(f'=========epoch{epoch}===========')

        writer.add_scalar(
                'Train/Loss',
                average_loss,
                epoch
            )

        writer.add_scalar(
                'Train/Perplexity',
                perplexity,
                epoch
            )
    generated_text = predict_ch8(prefix=predict_prefix,
                                     num_preds=num_preds,
                                     net = net,
                                     vocab = vocab,
                                     device=device)
    writer.add_text('预测文本', generated_text)
    print(f'预测文本：{generated_text}')
    writer.close()

##### 主函数

In [13]:
##加载数据,获取迭代器和字母数字映射
batch_size = 256
num_steps = 35

data_iter, vocab = load_data_time_machine(
    batch_size=batch_size,
    num_steps=num_steps,
    file_path='timemachine.txt',
)

##选择设备
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

num_hiddens = 512

##定义RNN网络
net = LSTM(
    vocab_size=len(vocab),
    num_hiddens=num_hiddens,
    device=device,
    get_params=get_params,
    init_state=init_lstm_state,
    forward_fn=lstm
)

##开始训练
num_epochs = 1000
logdir = './logs/lstm_first'
train_lstm(
    net=net,
    data_iter=data_iter,
    vocab=vocab,
    device=device,
    num_epochs=num_epochs,
    clip_theta=1.0,
    predict_prefix='time traveller',
    num_preds=30,log_dir=logdir
)

KeyboardInterrupt: 